# Gold — Cadastros mensais de clientes

Desenvolvido por: Ygor Moraes

## Objetivo

Criar a Gold `gold_ecommerce_clientes_cadastros_mensal`, medindo a evolução mensal de novos cadastros de clientes.

## Regra de negócio

Cada cliente entra no mês correspondente à sua `dt_cadastro`.

A Gold calcula:

- quantidade de novos clientes por mês;
- quantidade de novos clientes no mês anterior;
- crescimento percentual mês contra mês;
- quantidade acumulada de clientes.

## Fonte

- Silver `ecommerce_clientes`

## Cuidados técnicos

- A Silver `ecommerce_clientes` é lida como Delta.
- A Gold deve manter uma linha por mês de cadastro.
- Registros sem `dt_cadastro` não devem entrar corretamente na agregação mensal.

In [0]:
%run "../config/00_config"

In [0]:
%run "../utils/00_utils"

In [0]:
# Importa funções e define parâmetros da Gold.

from pyspark.sql.functions import (
    col,
    countDistinct,
    current_timestamp,
    lag,
    lit,
    month,
    round,
    sum as spark_sum,
    to_date,
    trunc,
    when,
    year
)

from pyspark.sql.window import Window

SILVER_CLIENTES_TABLE = "ecommerce_clientes"

SILVER_CLIENTES_PATH = f"{SILVER_BASE_PATH}{SILVER_CLIENTES_TABLE}"

GOLD_TABLE = "gold_ecommerce_clientes_cadastros_mensal"
GOLD_PATH = f"{GOLD_BASE_PATH}{GOLD_TABLE}"

FINAL_TABLE = f"{TARGET_SCHEMA}.{GOLD_TABLE}"

CLIENTES_REQUIRED_COLUMNS = [
    "id_cliente",
    "dt_cadastro"
]

GOLD_KEY_COLUMNS = [
    "ano_cadastro",
    "mes_cadastro",
    "data_referencia"
]

adls_options = get_adls_options()

print("Parâmetros definidos com sucesso.")
print("SILVER_CLIENTES_PATH:", SILVER_CLIENTES_PATH)
print("GOLD_PATH:", GOLD_PATH)
print("FINAL_TABLE:", FINAL_TABLE)

In [0]:
# Lê a Silver de clientes.

df_clientes = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(SILVER_CLIENTES_PATH)
)

total_clientes = df_clientes.count()

print("Silver de clientes lida com sucesso.")
print(f"Total clientes: {total_clientes}")

In [0]:
# Valida colunas obrigatórias, chave e data de cadastro.

validate_required_columns(df_clientes, CLIENTES_REQUIRED_COLUMNS)

clientes_distintos = (
    df_clientes
    .select(col("id_cliente").cast("int").alias("id_cliente"))
    .distinct()
    .count()
)

clientes_duplicados = total_clientes - clientes_distintos

clientes_sem_dt_cadastro = (
    df_clientes
    .filter(col("dt_cadastro").isNull())
    .count()
)

print(f"Total clientes: {total_clientes}")
print(f"Clientes distintos: {clientes_distintos}")
print(f"Clientes duplicados: {clientes_duplicados}")
print(f"Clientes sem dt_cadastro: {clientes_sem_dt_cadastro}")

if clientes_duplicados > 0:
    raise Exception("Erro: existem clientes duplicados por id_cliente.")

if clientes_sem_dt_cadastro > 0:
    raise Exception("Erro: existem clientes sem dt_cadastro.")

print("Validação OK: fonte mínima conferida.")

In [0]:
# Cria a Gold mensal de novos cadastros de clientes.

window_mensal = (
    Window
    .orderBy("ano_cadastro", "mes_cadastro")
)

df_gold = (
    df_clientes
    .select(
        col("id_cliente").cast("int").alias("id_cliente"),
        trunc(to_date(col("dt_cadastro")), "MM").alias("data_referencia"),
        year(col("dt_cadastro")).alias("ano_cadastro"),
        month(col("dt_cadastro")).alias("mes_cadastro")
    )
    .groupBy(
        "ano_cadastro",
        "mes_cadastro",
        "data_referencia"
    )
    .agg(
        countDistinct("id_cliente").alias("qtd_clientes_novos")
    )
    .withColumn(
        "qtd_clientes_novos_mes_anterior",
        lag("qtd_clientes_novos").over(window_mensal)
    )
    .withColumn(
        "crescimento_mom_percentual",
        round(
            when(
                col("qtd_clientes_novos_mes_anterior").isNotNull(),
                (
                    (col("qtd_clientes_novos") - col("qtd_clientes_novos_mes_anterior"))
                    / col("qtd_clientes_novos_mes_anterior")
                ) * 100
            ).otherwise(lit(None)),
            2
        )
    )
    .withColumn(
        "qtd_clientes_acumulado",
        spark_sum("qtd_clientes_novos").over(window_mensal)
    )
    .withColumn("gold_processed_at", current_timestamp())
    .orderBy("ano_cadastro", "mes_cadastro")
)

print("Gold de cadastros mensais criada em memória.")
display(df_gold)

In [0]:
# Valida totais, duplicidade de mês e campos principais da Gold.

total_linhas_gold = df_gold.count()

total_periodos_distintos = (
    df_gold
    .select(GOLD_KEY_COLUMNS)
    .distinct()
    .count()
)

periodos_duplicados = total_linhas_gold - total_periodos_distintos

validacao_gold = (
    df_gold
    .agg(
        spark_sum("qtd_clientes_novos").alias("total_clientes_novos"),
        spark_sum("qtd_clientes_acumulado").alias("soma_acumulado")
    )
    .collect()[0]
)

total_clientes_novos_gold = validacao_gold["total_clientes_novos"]

nulos_gold = (
    df_gold
    .filter(
        col("ano_cadastro").isNull() |
        col("mes_cadastro").isNull() |
        col("data_referencia").isNull() |
        col("qtd_clientes_novos").isNull() |
        col("qtd_clientes_acumulado").isNull()
    )
    .count()
)

ultimo_acumulado = (
    df_gold
    .orderBy(col("ano_cadastro").desc(), col("mes_cadastro").desc())
    .select("qtd_clientes_acumulado")
    .first()["qtd_clientes_acumulado"]
)

print(f"Total clientes fonte: {total_clientes}")
print(f"Total clientes novos na Gold: {total_clientes_novos_gold}")
print(f"Último acumulado: {ultimo_acumulado}")
print(f"Total linhas Gold: {total_linhas_gold}")
print(f"Períodos duplicados: {periodos_duplicados}")
print(f"Linhas com nulos principais: {nulos_gold}")

if total_clientes_novos_gold != total_clientes:
    raise Exception("Erro: total de clientes novos da Gold não fecha com a fonte.")

if ultimo_acumulado != total_clientes:
    raise Exception("Erro: acumulado final da Gold não fecha com a fonte.")

if periodos_duplicados > 0:
    raise Exception("Erro: existem períodos duplicados na Gold.")

if nulos_gold > 0:
    raise Exception("Erro: existem nulos nas colunas principais da Gold.")

print("Validação OK: Gold em memória conferida.")

In [0]:
# Grava a Gold em Delta no ADLS.

(
    df_gold
    .write
    .format("delta")
    .options(**adls_options)
    .option("overwriteSchema", "true")
    .mode("overwrite")
    .save(GOLD_PATH)
)

print(f"Gold gravada com sucesso em Delta: {GOLD_PATH}")

In [0]:
# Lê e valida a Gold Delta gravada.

df_gold_saved = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(GOLD_PATH)
)

total_linhas_gold_saved = df_gold_saved.count()

total_periodos_saved = (
    df_gold_saved
    .select(GOLD_KEY_COLUMNS)
    .distinct()
    .count()
)

periodos_duplicados_saved = total_linhas_gold_saved - total_periodos_saved

validacao_gold_saved = (
    df_gold_saved
    .agg(
        spark_sum("qtd_clientes_novos").alias("total_clientes_novos")
    )
    .collect()[0]
)

ultimo_acumulado_saved = (
    df_gold_saved
    .orderBy(col("ano_cadastro").desc(), col("mes_cadastro").desc())
    .select("qtd_clientes_acumulado")
    .first()["qtd_clientes_acumulado"]
)

print(f"Total linhas Gold Delta: {total_linhas_gold_saved}")
print(f"Períodos duplicados Gold Delta: {periodos_duplicados_saved}")
print(f"Total clientes fonte: {total_clientes}")
print(f"Total clientes Gold Delta: {validacao_gold_saved['total_clientes_novos']}")
print(f"Último acumulado Gold Delta: {ultimo_acumulado_saved}")

if validacao_gold_saved["total_clientes_novos"] != total_clientes:
    raise Exception("Erro: total de clientes da Gold Delta não confere.")

if ultimo_acumulado_saved != total_clientes:
    raise Exception("Erro: acumulado final da Gold Delta não confere.")

if periodos_duplicados_saved > 0:
    raise Exception("Erro: existem períodos duplicados na Gold Delta.")

print("Validação OK: Gold Delta gravada corretamente.")

In [0]:
# Prepara a Gold para escrita no SQL Server.

df_gold_sql = (
    df_gold_saved
    .select(
        col("ano_cadastro").cast("int").alias("ano_cadastro"),
        col("mes_cadastro").cast("int").alias("mes_cadastro"),
        col("data_referencia").cast("date").alias("data_referencia"),
        col("qtd_clientes_novos").cast("int").alias("qtd_clientes_novos"),
        col("qtd_clientes_novos_mes_anterior").cast("int").alias("qtd_clientes_novos_mes_anterior"),
        col("crescimento_mom_percentual").cast("decimal(10,2)").alias("crescimento_mom_percentual"),
        col("qtd_clientes_acumulado").cast("int").alias("qtd_clientes_acumulado"),
        col("gold_processed_at").cast("timestamp").alias("gold_processed_at")
    )
)

print("Gold preparada para escrita no SQL Server.")
df_gold_sql.printSchema()
display(df_gold_sql.orderBy("ano_cadastro", "mes_cadastro"))

In [0]:
# Grava a Gold diretamente na tabela final do SQL Server.

write_sql_table(
    df=df_gold_sql,
    sql_host=SQL_HOST,
    sql_database=SQL_DATABASE,
    sql_username=SQL_USERNAME,
    sql_password=SQL_PASSWORD,
    table_name=FINAL_TABLE,
    mode="overwrite",
    sql_port=SQL_PORT
)

print(f"Gold gravada com sucesso na tabela final: {FINAL_TABLE}")

In [0]:
# Lê e valida a tabela final do SQL Server.

df_final = read_sql_table(
    spark=spark,
    sql_host=SQL_HOST,
    sql_database=SQL_DATABASE,
    sql_username=SQL_USERNAME,
    sql_password=SQL_PASSWORD,
    table_name=FINAL_TABLE,
    sql_port=SQL_PORT
)

total_linhas_final = df_final.count()

total_periodos_final = (
    df_final
    .select(GOLD_KEY_COLUMNS)
    .distinct()
    .count()
)

periodos_duplicados_final = total_linhas_final - total_periodos_final

validacao_final = (
    df_final
    .agg(
        spark_sum("qtd_clientes_novos").alias("total_clientes_novos")
    )
    .collect()[0]
)

ultimo_acumulado_final = (
    df_final
    .orderBy(col("ano_cadastro").desc(), col("mes_cadastro").desc())
    .select("qtd_clientes_acumulado")
    .first()["qtd_clientes_acumulado"]
)

print(f"Total linhas tabela final: {total_linhas_final}")
print(f"Períodos duplicados tabela final: {periodos_duplicados_final}")
print(f"Total clientes fonte: {total_clientes}")
print(f"Total clientes tabela final: {validacao_final['total_clientes_novos']}")
print(f"Último acumulado tabela final: {ultimo_acumulado_final}")

if validacao_final["total_clientes_novos"] != total_clientes:
    raise Exception("Erro: total de clientes da tabela final não confere.")

if ultimo_acumulado_final != total_clientes:
    raise Exception("Erro: acumulado final da tabela final não confere.")

if periodos_duplicados_final > 0:
    raise Exception("Erro: existem períodos duplicados na tabela final.")

print("Validação OK: tabela final SQL Server gravada corretamente.")